# 35 - Magnetic Core MVP Tutorial

This notebook demonstrates the MVP magnetic-core contract:

- `component.magnetic_core` with `model: saturation`
- core-loss telemetry channel (`<component>.core_loss`)
- metadata inspection (`domain`, `unit`, `source_component`)


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'build' / 'python').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str((repo_root / 'build' / 'python').resolve()))

import pulsim as ps


In [2]:
netlist_path = repo_root / 'examples' / 'magnetic_core_saturation_freq_loss.yaml'
try:
    parser_opts = ps.YamlParserOptions()
    parser_opts.strict = False
    parser = ps.YamlParser(parser_opts)
except Exception:
    parser = ps.YamlParser()
circuit, opts = parser.load_string(netlist_path.read_text(encoding='utf-8'))
assert parser.errors == [], parser.errors

opts.newton_options.num_nodes = int(circuit.num_nodes())
opts.newton_options.num_branches = int(circuit.num_branches())
result = ps.Simulator(circuit, opts).run_transient(circuit.initial_state())
assert result.success, result.message

channel = 'Lsat.core_loss'
assert channel in result.virtual_channels
meta = result.virtual_channel_metadata[channel]
print('metadata:', meta.domain, meta.unit, meta.source_component)

t = [float(v) for v in result.time]
core_loss = [float(v) for v in result.virtual_channels[channel]]
print('avg core loss [W]:', sum(core_loss) / len(core_loss))
print('peak core loss [W]:', max(core_loss))
loss_rows = {row.device_name: row for row in result.loss_summary.device_losses}
if 'Lsat.core' in loss_rows:
    row = loss_rows['Lsat.core']
    print('summary row Lsat.core -> avg [W]:', float(row.average_power), 'energy [J]:', float(row.total_energy))

plt.figure(figsize=(10, 4))
plt.plot(t, core_loss, label=channel)
plt.xlabel('Time [s]')
plt.ylabel('Core Loss [W]')
plt.title('Magnetic Core Loss Telemetry (MVP)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


metadata: loss W Lsat
avg core loss [W]: 0.04620250445307797
peak core loss [W]: 0.08922605528012456
summary row Lsat.core -> avg [W]: 0.04623330612271347 energy [J]: 0.00013869991836813793


/var/folders/0n/gs7hh4fj4bn8r3r4qsykw2840000gn/T/ipykernel_84829/578442098.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
